# Rékolte — Notebook 2: Feature Engineering, Model Training & Evaluation
**Project:** Predicting Sugarcane Yield in Mauritius Using Satellite Imagery  
**Student:** Yashvin Booputh (M01006629)  
**Module:** CST3990 — Spring 2025/2026

---

## What this notebook does
1. Merges bulletin harvest data with GEE satellite features
2. Engineers the full feature matrix (15 features per row)
3. Trains Random Forest and XGBoost models with Leave-One-Season-Out (LOSO) cross-validation
4. Tunes hyperparameters using grid search
5. Evaluates both models on the 2025 holdout season
6. Saves trained models and data to Google Drive

### Feature matrix
| Features | Count |
|----------|-------|
| Monthly NDVI (June–December) | 7 |
| Season NDVI aggregates (mean, max, cumulative) | 3 |
| Region (one-hot: CENTRE, EST, NORD, OUEST vs SUD baseline) | 4 |
| Season year (trend proxy) | 1 |
| **Total** | **15** |

### Training / evaluation split
- **Training:** 2008–2024 (17 seasons × 5 regions = 85 rows) with LOSO CV
- **Holdout test:** 2025 (5 rows — never seen during training or tuning)

## Cell 1 — Install packages

In [ ]:
!pip install xgboost scikit-learn joblib --quiet
print('Packages ready.')

## Cell 2 — Imports & Drive mount

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE  = '/content/drive/MyDrive/CST3990 - Final Year Project'
DATA_DIR    = f'{DRIVE_BASE}/data'
MODEL_DIR   = f'{DRIVE_BASE}/model'
os.makedirs(MODEL_DIR, exist_ok=True)

print('Imports OK. Drive mounted.')

## Cell 3 — Load Data

In [ ]:
# Harvest bulletin data (90 rows: 18 seasons × 5 regions)
df_harvest = pd.read_csv(f'{DATA_DIR}/season_end_data.csv')
print(f'Harvest data: {df_harvest.shape}')
print(df_harvest[['season','region','tch']].head(10))

print()

# Satellite NDVI features (90 rows: 18 seasons × 5 regions)
df_sat = pd.read_csv(f'{MODEL_DIR}/satellite_features.csv')
print(f'Satellite features: {df_sat.shape}')
print(df_sat[['season','region','satellite','ndvi_mean','observation_count']].head(10))

## Cell 4 — Merge Datasets

In [ ]:
df = pd.merge(
    df_harvest[['season', 'region', 'tch', 'surface_harvested', 'cane_production']],
    df_sat,
    on=['season', 'region'],
    how='inner'
)

print(f'Merged dataset: {df.shape}')
print(f'Seasons: {sorted(df["season"].unique())}')
print(f'Regions: {sorted(df["region"].unique())}')
print(f'Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')

# Save merged features
df.to_csv(f'{MODEL_DIR}/merged_features.csv', index=False)
print(f'\nSaved merged_features.csv ({len(df)} rows)')

## Cell 5 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. TCH distribution
axes[0, 0].hist(df['tch'], bins=20, color='#2d5016', edgecolor='white')
axes[0, 0].set_title('TCH distribution (all regions/seasons)')
axes[0, 0].set_xlabel('TCH (tonnes cane / ha)')

# 2. TCH by region
regions = ['NORD', 'SUD', 'EST', 'OUEST', 'CENTRE']
tch_by_region = [df[df['region'] == r]['tch'].values for r in regions]
axes[0, 1].boxplot(tch_by_region, labels=regions, patch_artist=True)
axes[0, 1].set_title('TCH by region')
axes[0, 1].set_ylabel('TCH')

# 3. TCH over time
for region in regions:
    sub = df[df['region'] == region].sort_values('season')
    axes[0, 2].plot(sub['season'], sub['tch'], marker='o', label=region, linewidth=1.5)
axes[0, 2].set_title('TCH trends by region (2008–2025)')
axes[0, 2].set_xlabel('Season')
axes[0, 2].legend(fontsize=8)
axes[0, 2].tick_params(axis='x', rotation=45)

# 4. NDVI mean vs TCH
axes[1, 0].scatter(df['ndvi_mean'], df['tch'], c='#C8891A', alpha=0.7)
axes[1, 0].set_xlabel('NDVI mean (season)')
axes[1, 0].set_ylabel('TCH')
axes[1, 0].set_title('NDVI mean vs TCH')

# 5. NDVI cumulative vs TCH
axes[1, 1].scatter(df['ndvi_cumulative'], df['tch'], c='#2d5016', alpha=0.7)
axes[1, 1].set_xlabel('NDVI cumulative')
axes[1, 1].set_ylabel('TCH')
axes[1, 1].set_title('NDVI cumulative vs TCH')

# 6. Correlation heatmap (NDVI features + TCH)
ndvi_cols = [c for c in df.columns if c.startswith('ndvi_')]
corr_cols = ndvi_cols + ['season', 'tch']
corr = df[corr_cols].corr()
sns.heatmap(corr[['tch']].sort_values('tch', ascending=False),
            annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, ax=axes[1, 2],
            cbar_kws={'shrink': 0.8})
axes[1, 2].set_title('Correlation with TCH')

plt.suptitle('Exploratory Data Analysis — Rékolte', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/eda_plots.png', dpi=150)
plt.show()

## Cell 6 — Feature Engineering

1. One-hot encode `region`
2. Impute missing monthly NDVI values with that row's season mean NDVI  
   (only affects months with zero cloud-free observations)
3. Split into train (2008–2024) and holdout test (2025)

In [ ]:
# ── Impute missing monthly NDVI ────────────────────────────────────────────────
monthly_cols = ['ndvi_june','ndvi_july','ndvi_aug','ndvi_sep','ndvi_oct','ndvi_nov','ndvi_dec']

for col in monthly_cols:
    missing = df[col].isnull().sum()
    if missing > 0:
        # Fill with season mean for that row
        df[col] = df[col].fillna(df['ndvi_mean'])
        print(f'Imputed {missing} missing values in {col} with row ndvi_mean')

# ── One-hot encode region ──────────────────────────────────────────────────────
df_encoded = pd.get_dummies(df, columns=['region'], drop_first=True, dtype=int)
# drop_first=True drops 'region_CENTRE' → baseline is CENTRE

region_dummies = [c for c in df_encoded.columns if c.startswith('region_')]
print(f'Region dummies: {region_dummies}')

# ── Define feature columns ────────────────────────────────────────────────────
NDVI_FEATURES = monthly_cols + ['ndvi_mean', 'ndvi_max', 'ndvi_cumulative']
ALL_FEATURES  = NDVI_FEATURES + region_dummies + ['season']
TARGET        = 'tch'

print(f'\nTotal features: {len(ALL_FEATURES)}')
print(f'Features: {ALL_FEATURES}')

# ── Train / holdout split ─────────────────────────────────────────────────────
train_df = df_encoded[df_encoded['season'] <= 2024].copy()
test_df  = df_encoded[df_encoded['season'] == 2025].copy()

X_train = train_df[ALL_FEATURES].values
y_train = train_df[TARGET].values
X_test  = test_df[ALL_FEATURES].values
y_test  = test_df[TARGET].values

print(f'\nTraining set: {X_train.shape}  ({len(train_df["season"].unique())} seasons)')
print(f'Test set:     {X_test.shape}   (2025 holdout)')

## Cell 7 — Leave-One-Season-Out (LOSO) Cross-Validation Setup

Standard k-fold CV would leak future seasons into training folds — wrong for time-structured data.  
LOSO holds out one entire season at a time:
- Train on 16 seasons → predict 1 held-out season → rotate (17 iterations)
- This simulates realistic "predict next season" performance.

In [ ]:
def loso_evaluate(model, X, y, seasons_array):
    """
    Leave-One-Season-Out evaluation.
    Returns dict with per-fold predictions and overall RMSE, MAE, R².
    """
    unique_seasons = sorted(np.unique(seasons_array))
    all_preds  = np.zeros(len(y))
    all_actual = y.copy()

    for held_out in unique_seasons:
        train_idx = seasons_array != held_out
        test_idx  = seasons_array == held_out

        model.fit(X[train_idx], y[train_idx])
        preds = model.predict(X[test_idx])
        all_preds[test_idx] = preds

    rmse = np.sqrt(mean_squared_error(all_actual, all_preds))
    mae  = mean_absolute_error(all_actual, all_preds)
    r2   = r2_score(all_actual, all_preds)

    return {
        'predictions': all_preds,
        'actual':      all_actual,
        'rmse': rmse,
        'mae':  mae,
        'r2':   r2
    }

# Seasons array for LOSO
train_seasons = train_df['season'].values
print('LOSO setup ready.')
print(f'Seasons in training set: {sorted(np.unique(train_seasons))}')

## Cell 8 — Random Forest: Grid Search + LOSO Evaluation

Grid search finds the best hyperparameters.  
We use LOSO CV as the scoring method so the grid search itself is cross-validated properly.

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.model_selection import GridSearchCV

logo = LeaveOneGroupOut()

# ── Random Forest grid search ─────────────────────────────────────────────────
rf_param_grid = {
    'n_estimators':      [100, 300, 500],
    'max_depth':         [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', 0.5]
}

rf_base = RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1)
rf_grid = GridSearchCV(
    rf_base,
    rf_param_grid,
    cv=logo,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
rf_grid.fit(X_train, y_train, groups=train_seasons)

best_rf_params = rf_grid.best_params_
print(f'\nBest RF params: {best_rf_params}')
print(f'Best CV RMSE:   {-rf_grid.best_score_:.4f}')

# ── LOSO evaluation with best params ─────────────────────────────────────────
best_rf = RandomForestRegressor(**best_rf_params, random_state=RANDOM_SEED, n_jobs=-1)
rf_loso = loso_evaluate(best_rf, X_train, y_train, train_seasons)

print(f'\nRandom Forest — LOSO CV Results:')
print(f'  RMSE: {rf_loso["rmse"]:.4f}')
print(f'  MAE:  {rf_loso["mae"]:.4f}')
print(f'  R²:   {rf_loso["r2"]:.4f}')

## Cell 9 — XGBoost: Grid Search + LOSO Evaluation

In [ ]:
xgb_param_grid = {
    'n_estimators':  [100, 300, 500],
    'max_depth':     [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample':     [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5]
}

xgb_base = XGBRegressor(
    random_state=RANDOM_SEED,
    tree_method='hist',
    verbosity=0
)
xgb_grid = GridSearchCV(
    xgb_base,
    xgb_param_grid,
    cv=logo,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
xgb_grid.fit(X_train, y_train, groups=train_seasons)

best_xgb_params = xgb_grid.best_params_
print(f'\nBest XGBoost params: {best_xgb_params}')
print(f'Best CV RMSE:        {-xgb_grid.best_score_:.4f}')

best_xgb = XGBRegressor(**best_xgb_params, random_state=RANDOM_SEED, tree_method='hist', verbosity=0)
xgb_loso = loso_evaluate(best_xgb, X_train, y_train, train_seasons)

print(f'\nXGBoost — LOSO CV Results:')
print(f'  RMSE: {xgb_loso["rmse"]:.4f}')
print(f'  MAE:  {xgb_loso["mae"]:.4f}')
print(f'  R²:   {xgb_loso["r2"]:.4f}')

## Cell 10 — Compare LOSO CV Results

In [ ]:
print('=' * 45)
print(f'{"Metric":<12} {"Random Forest":>15} {"XGBoost":>15}')
print('=' * 45)
for metric in ['rmse', 'mae', 'r2']:
    print(f'{metric.upper():<12} {rf_loso[metric]:>15.4f} {xgb_loso[metric]:>15.4f}')
print('=' * 45)

# Scatter: predicted vs actual (LOSO)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, result, name, color in [
    (axes[0], rf_loso,  'Random Forest', '#2d5016'),
    (axes[1], xgb_loso, 'XGBoost',       '#C8891A')
]:
    lo = min(result['actual'].min(), result['predictions'].min()) - 2
    hi = max(result['actual'].max(), result['predictions'].max()) + 2
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1, alpha=0.5, label='Perfect')
    ax.scatter(result['actual'], result['predictions'], color=color, alpha=0.7, s=50)
    ax.set_xlabel('Actual TCH')
    ax.set_ylabel('Predicted TCH')
    ax.set_title(f'{name} — LOSO CV\nRMSE={result["rmse"]:.2f}  MAE={result["mae"]:.2f}  R²={result["r2"]:.3f}')
    ax.legend()

plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/loso_predicted_vs_actual.png', dpi=150)
plt.show()

## Cell 11 — Train Final Models on Full 2008–2024 Training Set

In [ ]:
# Train on ALL 85 rows (2008-2024) using the best hyperparameters found above
final_rf = RandomForestRegressor(**best_rf_params, random_state=RANDOM_SEED, n_jobs=-1)
final_rf.fit(X_train, y_train)

final_xgb = XGBRegressor(**best_xgb_params, random_state=RANDOM_SEED,
                          tree_method='hist', verbosity=0)
final_xgb.fit(X_train, y_train)

print('Final models trained on 2008–2024.')

## Cell 12 — Evaluate on 2025 Holdout
These 5 rows (one per region) were **never seen** during training or hyperparameter tuning.

In [ ]:
rf_pred_2025  = final_rf.predict(X_test)
xgb_pred_2025 = final_xgb.predict(X_test)

test_regions = test_df['region'].values if 'region' in test_df.columns else [
    r for r in ['NORD','SUD','EST','OUEST','CENTRE']]

results_2025 = pd.DataFrame({
    'region':        test_df['region'].values if 'region' in test_df.columns else regions,
    'actual_tch':    y_test,
    'rf_predicted':  rf_pred_2025,
    'xgb_predicted': xgb_pred_2025,
    'rf_error':      rf_pred_2025  - y_test,
    'xgb_error':     xgb_pred_2025 - y_test
})

print('2025 Holdout Results:')
print(results_2025.to_string(index=False, float_format='%.2f'))

print('\n2025 Holdout Metrics:')
for name, preds in [('Random Forest', rf_pred_2025), ('XGBoost', xgb_pred_2025)]:
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae  = mean_absolute_error(y_test, preds)
    r2   = r2_score(y_test, preds)
    print(f'  {name:<20}  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}')

results_2025.to_csv(f'{MODEL_DIR}/predictions_2025.csv', index=False)
print('\nSaved predictions_2025.csv')

## Cell 13 — 2025 Holdout Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(results_2025))
width = 0.25

for ax, pred_col, title, color in [
    (axes[0], 'rf_predicted',  'Random Forest — 2025 Holdout', '#2d5016'),
    (axes[1], 'xgb_predicted', 'XGBoost — 2025 Holdout',       '#C8891A')
]:
    bars1 = ax.bar(x - width/2, results_2025['actual_tch'],  width, label='Actual',    color='#888', alpha=0.7)
    bars2 = ax.bar(x + width/2, results_2025[pred_col],      width, label='Predicted', color=color,  alpha=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels(results_2025['region'])
    ax.set_ylabel('TCH (tonnes cane / ha)')
    ax.set_title(title)
    ax.legend()
    ax.set_ylim(0, max(results_2025['actual_tch'].max(), results_2025[pred_col].max()) * 1.15)

    # Annotate error
    for i, (actual, pred) in enumerate(zip(results_2025['actual_tch'], results_2025[pred_col])):
        err = pred - actual
        ax.text(i + width/2, pred + 1, f'{err:+.1f}', ha='center', fontsize=8, color='red' if abs(err) > 5 else 'green')

plt.suptitle('2025 Actual vs Predicted TCH by Region', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/predictions_2025_bar.png', dpi=150)
plt.show()

## Cell 14 — Feature Importance
Which NDVI months and features matter most for predicting TCH?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, model, name, color in [
    (axes[0], final_rf,  'Random Forest', '#2d5016'),
    (axes[1], final_xgb, 'XGBoost',       '#C8891A')
]:
    importances = model.feature_importances_
    feat_imp = pd.Series(importances, index=ALL_FEATURES).sort_values(ascending=True)

    bars = ax.barh(feat_imp.index, feat_imp.values, color=color, alpha=0.85)
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'{name} Feature Importance')

    # Label values
    for bar, val in zip(bars, feat_imp.values):
        ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/feature_importance.png', dpi=150)
plt.show()

print('Top 5 features (Random Forest):')
rf_fi = pd.Series(final_rf.feature_importances_, index=ALL_FEATURES).sort_values(ascending=False)
print(rf_fi.head(5).to_string())

print('\nTop 5 features (XGBoost):')
xgb_fi = pd.Series(final_xgb.feature_importances_, index=ALL_FEATURES).sort_values(ascending=False)
print(xgb_fi.head(5).to_string())

## Cell 15 — Save Models and Summary

In [ ]:
import json as json_lib

RF_PATH  = f'{MODEL_DIR}/rf_model.joblib'
XGB_PATH = f'{MODEL_DIR}/xgb_model.joblib'

joblib.dump(final_rf,  RF_PATH)
joblib.dump(final_xgb, XGB_PATH)

print(f'Saved: {RF_PATH}')
print(f'Saved: {XGB_PATH}')

# Save model metadata for Flask API
rf_2025_rmse  = float(np.sqrt(mean_squared_error(y_test, rf_pred_2025)))
xgb_2025_rmse = float(np.sqrt(mean_squared_error(y_test, xgb_pred_2025)))

metadata = {
    'feature_columns': ALL_FEATURES,
    'target': TARGET,
    'training_seasons': list(range(2008, 2025)),
    'test_season': 2025,
    'satellite_strategy': {
        '2008-2012': 'Landsat 7 ETM+ (USGS/LE07/C02/T1_L2)',
        '2013-2016': 'Landsat 8 OLI (USGS/LC08/C02/T1_L2)',
        '2017-2025': 'Sentinel-2 SR Harmonized (COPERNICUS/S2_SR_HARMONIZED)'
    },
    'random_seed': RANDOM_SEED,
    'models': {
        'random_forest': {
            'params':      best_rf_params,
            'loso_rmse':   rf_loso['rmse'],
            'loso_mae':    rf_loso['mae'],
            'loso_r2':     rf_loso['r2'],
            'test_2025_rmse': rf_2025_rmse
        },
        'xgboost': {
            'params':      best_xgb_params,
            'loso_rmse':   xgb_loso['rmse'],
            'loso_mae':    xgb_loso['mae'],
            'loso_r2':     xgb_loso['r2'],
            'test_2025_rmse': xgb_2025_rmse
        }
    }
}

with open(f'{MODEL_DIR}/model_metadata.json', 'w') as f:
    json_lib.dump(metadata, f, indent=2)

print(f'Saved: {MODEL_DIR}/model_metadata.json')

print('\n' + '=' * 50)
print('FINAL SUMMARY')
print('=' * 50)
print(f'{"":25} {"RF":>10} {"XGBoost":>10}')
print(f'{"LOSO RMSE (CV)":<25} {rf_loso["rmse"]:>10.4f} {xgb_loso["rmse"]:>10.4f}')
print(f'{"LOSO MAE  (CV)":<25} {rf_loso["mae"]:>10.4f} {xgb_loso["mae"]:>10.4f}')
print(f'{"LOSO R²   (CV)":<25} {rf_loso["r2"]:>10.4f} {xgb_loso["r2"]:>10.4f}')
print(f'{"2025 Test RMSE":<25} {rf_2025_rmse:>10.4f} {xgb_2025_rmse:>10.4f}')
print('=' * 50)

## Cell 16 — Files saved to Google Drive
Verify everything is in place before proceeding to the Flask API.

In [ ]:
print(f'Files in {MODEL_DIR}:')
for f in sorted(os.listdir(MODEL_DIR)):
    size_kb = os.path.getsize(f'{MODEL_DIR}/{f}') / 1024
    print(f'  {f:<40}  {size_kb:>8.1f} KB')

print('\nNext step: Flask API — load rf_model.joblib + xgb_model.joblib, serve /predict endpoint.')